![Practical AI 101](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/main/week2/assets/thumbnail.png)

![Practical AI 101](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/main/week2/assets/thumbnail2.png)

# ⚠️ 실습 전 필수: 본인 드라이브에 사본 만들기!

상단 메뉴 **파일 → 드라이브에 사본 저장**을 클릭하세요.

사본을 만들지 않으면 작성한 코드가 저장되지 않습니다. 꼭 먼저 사본을 만든 뒤 시작하세요!

## 2주차 · LLM API 와 Agent 기초 (Responses API / Chat Completions)

지난 주에는 PyTorch로 딥러닝의 기본기를 다졌습니다.
이번 주에는 이미 완성되어 있는 거대 언어 모델(LLM)을 **API로 빌려 쓰는 법**과,
모델이 스스로 도구를 골라 쓰며 문제를 해결하는 **에이전트(Agent)** 를 직접 만들어 봅니다.

### API가 뭔가요? — 식당에 비유하면

GPT 같은 대형 모델은 너무 커서 개인 PC나 Colab에서 직접 돌릴 수 없습니다.
그래서 OpenAI·구글 등은 모델을 자기 서버에 두고, **인터넷으로 주문을 받아 결과를 돌려주는 창구**를 열어 둡니다.
이 창구가 **API**(Application Programming Interface)입니다.

> 🍽️ 우리는 **주문서**(프롬프트)를 내고 **요리**(답변)를 받습니다.
> 주방(GPU, 모델 내부)이 어떻게 생겼는지는 몰라도 됩니다.
> 대신 **주문한 만큼 계산합니다** — LLM API는 주고받은 글자 양(토큰)에 비례해 과금됩니다.

1주차가 요리를 직접 배우는 시간이었다면, 오늘은 **잘 차려진 주방을 주문서 한 장으로 쓰는** 시간입니다.

### 오늘 쓰는 환경

- 🧪 **직접 실습** → [OpenRouter](https://openrouter.ai)의 **무료 모델**을 씁니다. (카드 등록 불필요)
- 📖 **참고** → 같은 일을 OpenAI 유료 [**Responses API**](https://platform.openai.com/docs/api-reference/responses)로 하면 어떻게 되는지도 잠깐 비교합니다.

둘은 겉모습만 조금 다를 뿐 **핵심 개념은 완전히 같습니다.**
오늘 무료 환경에서 배운 것은 나중에 어떤 API로든 그대로 옮겨 쓸 수 있습니다.

### 오늘의 목표

1. 무료 API 키 발급받기
2. LLM에 첫 요청 보내고 응답 받기
3. 대화 맥락 이어가기
4. 모델에게 **도구(함수)** 쥐여 주기 — function calling
5. 도구를 알아서 골라 쓰는 **에이전트** 직접 만들기

### 오늘의 최종 목표 미리 보기

```
[나] "서울 날씨 알려주고 25×4도 계산해줘"
   │
   ▼
[LLM]  "get_weather(city='서울') 를 실행해줘"   ← 모델은 실행을 '요청'만 한다
   │
   ▼
[내 코드] get_weather("서울") 실행 → "맑음, 27도"   ← 실제 실행은 우리가 한다
   │
   ▼
[LLM]  "calculate('25*4') 도 실행해줘"  → 100
   │
   ▼
[LLM]  "서울은 맑고 27도이며, 25×4는 100입니다."   ← 더 부를 도구가 없으면 최종 답변
```

이 왕복을 **함수 호출이 없어질 때까지 반복하는 루프**가 곧 에이전트입니다. 오늘 마지막에 직접 만듭니다.

---
## 1. OpenRouter 가입 & API 키 발급

[**OpenRouter**](https://openrouter.ai) 는 여러 회사의 LLM(OpenAI·구글·메타 등)을
**하나의 창구로 골라 쓸 수 있게 해주는 중계 서비스**입니다. 일부 모델은 무료라 오늘 실습에 딱 좋습니다.

### 키 발급 절차 (약 2분)

1. [openrouter.ai](https://openrouter.ai) 접속 → 우측 상단 **Sign In**
2. **Google** 또는 **GitHub** 계정으로 로그인 (별도 회원가입 불필요)
3. [openrouter.ai/keys](https://openrouter.ai/keys) 로 이동 (또는 프로필 → **Keys**)
4. **Create Key** 클릭 → 이름(예: `week2-practice`) 입력 → **Create**
5. 생성된 키(`sk-or-v1-...`)를 **지금 바로 복사**하세요. ⚠️ 이 화면을 벗어나면 다시 볼 수 없습니다.

> 💳 `:free` 가 붙은 모델은 결제 없이 쓰는 대신 횟수 제한이 있습니다(대략 **분당 20회 / 하루 200회**).
> 오늘 실습에는 충분합니다.

---
## 2. 환경 설정

준비는 세 단계입니다.

1. `openai` 라이브러리 설치
2. API 키 입력 (코드에 직접 적지 않기!)
3. `client` 객체 만들기 — 앞으로 모든 요청은 이 창구를 통해 나갑니다

OpenRouter는 **OpenAI 파이썬 라이브러리와 호환**됩니다. 접속 주소(`base_url`)와 키만 바꾸면
같은 코드가 그대로 동작하므로, 나중에 유료 OpenAI 키가 생겨도 두 줄만 고치면 됩니다.

> `!pip install` 맨 앞의 `!` 는 "파이썬이 아니라 터미널 명령으로 실행하라"는 Colab 문법입니다.
> Colab에 기본 설치된 `openai` 는 버전이 낮을 수 있어 매번 최신으로 올립니다.

In [ ]:
!pip install -q --upgrade openai

### API 키 입력

방금 복사한 키를 파이썬에게 알려 줍니다.

> 🔐 **키를 코드에 직접 쓰지 않는 이유**
> API 키는 **비밀번호이자 신용카드**입니다. 노트북을 공유하면 키도 함께 공유됩니다.
> 그래서 키는 코드가 아닌 **환경변수**라는 별도 보관함에 넣는 것이 원칙입니다.

아래 셀을 실행하면 키를 물어봅니다. 붙여넣고 Enter를 누르세요.
(입력한 글자는 화면에 표시되지 않습니다 — 화면 공유 중에도 안전합니다.)

> 💡 Colab 왼쪽 사이드바 🔑(보안 비밀)에 `OPENROUTER_API_KEY` 로 저장해 두면 매번 입력하지 않아도 됩니다.

In [ ]:
import os
from getpass import getpass    # 입력한 글자를 화면에 표시하지 않는 안전한 입력 함수

if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        # Colab 왼쪽 🔑(보안 비밀) 메뉴에 저장해 뒀다면 그 값을 사용
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        # 저장해 둔 게 없거나 Colab이 아니면 직접 입력
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API 키(sk-or-v1-...)를 입력하세요: ")

print("API 키 설정 완료 ✅")   # 키 자체는 절대 print 하지 않습니다

### 클라이언트 만들기

`client` 는 어느 서버에 어떤 키로 접속할지 담아 두는 객체입니다. 한 번 만들면 계속 재사용합니다.

- `base_url` 을 OpenRouter 주소로 바꿨기 때문에 요청이 OpenRouter로 갑니다. (이 줄이 없으면 OpenAI 본사로 갑니다)
- 모델 이름 끝의 `:free` 는 무료 모델 표시입니다. `MODEL` 상수 한 줄만 고치면 모델을 바꿀 수 있습니다.

In [ ]:
from openai import OpenAI

client = OpenAI(
    # base_url 만 바꾸면 같은 코드가 OpenRouter로 나갑니다. (지우면 OpenAI 본사)
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

---
## 3. 첫 번째 응답 받기 (Chat Completions)

기본형은 `client.chat.completions.create(...)` 입니다.
**대화 내용을 `messages` 리스트로 보내고**, 답은 `response.choices[0].message.content` 로 꺼냅니다.

### `messages` — 대화의 대본

각 항목은 "누가(`role`) · 무슨 말을(`content`)" 한 쌍입니다.

| role | 누구? | 언제 쓰나 |
|---|---|---|
| `"system"` | 모델에게 주는 설정·규칙 | 말투·역할·규칙 정하기 (5절) |
| `"user"` | 나 | 실제 질문·요청 |
| `"assistant"` | 모델의 지난 답변 | 대화 이어가기 (6절) |

### 꼭 알아둘 것: 서버는 대화를 기억하지 않습니다

모델은 **매번 처음 보는 사람처럼** 요청을 받습니다(stateless).
그래서 대화를 이어가려면 이전 대화까지 `messages` 에 담아 매번 다시 보내야 합니다. (6절에서 직접 해봅니다)

> `choices[0]` 인 이유: 답변 후보를 여러 개 받을 수도 있어 목록으로 옵니다. 보통 1개라 항상 `[0]` 을 씁니다.

<!-- manim-visual -->
### 🎬 시각 자료: API 요청과 응답 한 번 왕복

내 노트북이 `messages` 를 담아 요청을 보내면 서버의 모델이 답변 하나를 돌려줍니다. 서버는 이 호출을 기억하지 않으며, 주고받은 토큰 수만큼 비용이 발생합니다.

![ApiRoundTrip](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/main/week2/assets/ApiRoundTrip.gif)

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "자연어 처리에 대해 전혀 모르는 사람도 이해할 수 있게 두 문장으로 설명해줘."},
    ],
)

print(response.choices[0].message.content)

# 💡 content 를 바꿔 다른 질문으로도 실습해 보세요.
#    "딥러닝이 뭔지 초등학생도 이해할 수 있게 두 문장으로 설명해줘."
#    "다음 문장의 맞춤법을 고쳐줘: '오늘 회의는 3시에 시작되요.'"
#    "파이썬 리스트와 튜플의 차이를 표로 정리해줘."

방금 받은 `response` 안에는 답변 말고도 정보가 더 들어 있습니다. 두 가지만 봅니다.

- **`response.model`** — 실제로 답한 모델 이름. (요청한 모델이 붐비면 다른 버전이 대신 답하기도 합니다)
- **`response.usage`** — **토큰 사용량 = 요금 미터기.**
  LLM은 글자가 아니라 **토큰**(단어 조각) 단위로 텍스트를 세고, 요금도 토큰 수로 매깁니다.
  - `prompt_tokens` : 내가 보낸 입력의 양
  - `completion_tokens` : 모델이 생성한 답변의 양
  - `total_tokens` : 둘의 합 = 이번 호출의 비용

> 📏 한국어는 대략 1글자 ≈ 1~2토큰.
> **대화가 길어질수록 매번 보내는 양이 늘어 요금도 늘어납니다.** 6절에서 숫자로 확인합니다.

In [ ]:
print("사용 모델:", response.model)          # 실제로 답한 모델 (요청한 것과 다를 수 있음)
print("토큰 사용량:", response.usage)        # prompt(입력) / completion(출력) / total
print("입력 토큰:", response.usage.prompt_tokens)

---
## 4. 📖 [참고] OpenAI Responses API — 유료 키가 있다면

**Responses API**는 OpenAI의 최신 인터페이스로, 같은 일을 더 단순하게 처리합니다.
**(OpenRouter에서는 지원하지 않으므로 오늘은 눈으로만 봐 두세요.)**

```python
from openai import OpenAI
client = OpenAI(api_key="OpenAI 키")          # base_url 없음 = OpenAI 본사

resp = client.responses.create(
    model="gpt-4.1",
    input="딥러닝을 두 문장으로 설명해줘.",
)
print(resp.output_text)

# 대화 이어가기는 이전 응답 id 한 줄이면 끝 — 서버가 대화를 기억해 줍니다
resp2 = client.responses.create(
    model="gpt-4.1",
    input="방금 설명을 한 문장으로 줄여줘.",
    previous_response_id=resp.id,             # 👈 Chat Completions에는 없는 기능
)
print(resp2.output_text)
```

### 한눈에 보는 차이

| | **Chat Completions** (오늘 실습) | **Responses API** (OpenAI) |
|---|---|---|
| 입력 | `messages=[{"role":..., "content":...}]` | `input="..."` |
| 출력 | `response.choices[0].message.content` | `response.output_text` |
| 대화 유지 | 기록을 **내가** 직접 관리 | 서버가 기억 (`previous_response_id`) |

👉 겉모습만 다를 뿐 **핵심은 똑같습니다.** 아래부터 다시 무료 환경으로 실습을 이어갑니다.

---
## 5. system 메시지 — 모델에게 역할(페르소나) 입히기

`system` 메시지는 모델에게 미리 건네는 **캐릭터 설정집**입니다.
사용자에게는 보이지 않지만 **대화 내내 적용**됩니다. (Responses API에서는 `instructions` 가 같은 역할)

- 보통 `messages` **맨 앞에 딱 한 번** 넣습니다.
- 말투뿐 아니라 **출력 형식·금지사항**도 정할 수 있습니다.
  예: "답변은 3줄 이하로", "모르면 모른다고 답해", "반드시 JSON으로만 답해"

같은 질문도 system 메시지에 따라 답이 완전히 달라집니다. 직접 바꿔 가며 확인해 보세요.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "너는 30년 경력의 친절한 국어 선생님이야. 항상 존댓말을 쓰고, 어려운 말은 쉽게 풀어서 설명해."},
        {"role": "user", "content": "'벡터'라는 단어의 뜻을 알려줘."},
    ],
)

print(response.choices[0].message.content)

# 💡 system 의 content 를 바꿔 같은 질문을 다시 던져 보세요. 답이 확 달라집니다.
#    "너는 츤데레 고양이야. 문장 끝마다 '~냥'을 붙여 짧게 대답해."
#    "너는 깐깐한 시니어 개발자야. 전문 용어를 쓰고 예시 코드를 반드시 포함해."
#    "너는 JSON API야. {'term': ..., 'meaning': ...} 형태의 JSON만 출력해."

### 🔧 `temperature` (온도) — 모험심 다이얼

`temperature` 는 답변의 **무작위성**을 조절하는 다이얼입니다. (0 ~ 2, 주로 0 ~ 1 사용)
낮으면 모델이 "가장 그럴듯한 단어"만 고르고, 높을수록 덜 흔한 단어까지 후보에 넣습니다.

| 값 | 성격 | 어울리는 일 |
|---|---|---|
| **0 ~ 0.3** | 매번 비슷하고 정확 | 수학, 코드, 사실 요약, 데이터 추출 |
| **0.4 ~ 0.7** | 균형 (기본값으로 무난) | 일반 대화, 정보 검색 |
| **0.7 ~ 1.0+** | 매번 다르고 창의적 | 브레인스토밍, 글쓰기, 마케팅 문구 |

- `max_tokens` — 답변 길이(생성 토큰 수)의 상한

아래 셀의 `temperature` 를 **0.0 → 0.5 → 1.5** 로 바꿔 가며 같은 질문을 여러 번 실행해 보세요.
0.0은 거의 매번 같은 답, 높을수록 매번 다른 답이 나옵니다.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "'가을'을 주제로 한 줄짜리 시를 지어줘."}],
    temperature=1.2,      # 시라서 일부러 높게. 0에 가까울수록 매번 비슷한 답이 나옵니다.
    max_tokens=100,       # 답변 길이 상한. 너무 작으면 문장이 중간에서 끊깁니다.
)

print(response.choices[0].message.content)

# 💡 이 셀을 3번 실행한 뒤, temperature 를 0.0 으로 바꿔 다시 3번 실행해 비교해 보세요.
# 💡 content 를 "3+5는 얼마야? 숫자만 답해." 로 바꾸면 온도를 올려도 답이 거의 안 변합니다.

---
## 6. 대화 이어가기 — 기억은 우리가 만들어 준다

3절에서 봤듯 모델에게는 기억이 없습니다.
**매 요청에 담아 보낸 `messages` 가 모델이 아는 전부**입니다.
그래서 대화를 이어가려면 기록을 이렇게 쌓아서 보냅니다.

```
1턴: [user1]                                    → 답변1
2턴: [user1, assistant1, user2]                 → 답변2
3턴: [user1, assistant1, user2, assistant2, user3] → 답변3
```

핵심: **모델의 답변(`assistant`)도 리스트에 다시 넣어 줘야** 다음 턴에서 맥락이 이어집니다.

> 💸 대화가 길어질수록 매번 보내는 양(= `prompt_tokens` = 요금)도 늘어납니다.
> 그래서 아주 긴 대화는 오래된 부분을 잘라내거나 요약해서 보냅니다.
>
> 📖 Responses API라면 이 절 전체가 `previous_response_id` **한 줄**로 끝납니다. 서버가 대신 기억해 주니까요.

<!-- manim-visual -->
### 🎬 시각 자료: 대화 맥락은 매번 다시 보낸다

턴이 늘어날 때마다 `messages` 리스트 전체를 다시 전송합니다. 모델의 답변(assistant)까지 쌓아야 맥락이 이어지고, 그만큼 prompt 토큰과 비용도 함께 늘어납니다.

![MessagesContext](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/main/week2/assets/MessagesContext.gif)

In [ ]:
messages = [
    {"role": "user", "content": "내가 좋아하는 숫자는 7이야. 기억해둬."},
]

first = client.chat.completions.create(model=MODEL, messages=messages)
answer1 = first.choices[0].message.content
print("1차 응답:", answer1)

# ⭐ 모델의 답변도 기록에 넣어야 다음 턴에서 맥락이 유지됩니다. (누락하기 쉬운 부분)
messages.append({"role": "assistant", "content": answer1})

In [ ]:
# 새 질문을 덧붙이고 '기록 전체'를 다시 보냅니다. 이번 질문엔 '7'이 없지만 앞 대화로 알아냅니다.
messages.append({"role": "user", "content": "내가 좋아하는 숫자에 3을 곱하면 얼마야?"})

second = client.chat.completions.create(model=MODEL, messages=messages)
print("2차 응답:", second.choices[0].message.content)
print("이번 요청의 입력 토큰:", second.usage.prompt_tokens)   # 1턴보다 늘어난 만큼이 추가 비용

### 확인 실습

기록 없이 두 번째 질문만 보내면 어떻게 될까요?

```python
# 기록을 빼고 두 번째 질문만 단독으로 보내기
alone = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "내가 좋아하는 숫자에 3을 곱하면 얼마야?"}],
)
print(alone.choices[0].message.content)   # "어떤 숫자인지 알려주세요" 같은 답이 나옵니다
```

**모델의 기억은 요청마다 우리가 만들어 보내는 것** — 오늘 가장 중요한 개념 중 하나입니다.

---
## 7. 함수 호출(Function Calling) — 모델에게 도구 쥐여 주기

LLM은 아는 것도 많고 말도 잘하지만, **인터넷도 계산기도 없는 방에 갇힌 똑똑한 사람**과 같습니다.

- 오늘 날씨, 지금 환율, 우리 회사 DB → **모릅니다**
- 큰 수의 정확한 계산 → 그럴듯하게 **틀립니다** (환각, hallucination)
- 메일 보내기, 파일 저장 → **행동할 손이 없습니다**

### 해결책: 도구를 쥐여 주자

우리가 파이썬 함수를 만들어 "이런 도구가 있어"라고 알려 주면,
모델은 필요한 순간에 **"그 도구를 이 값으로 실행해 줘"라고 요청**합니다.

> ⚠️ **오늘의 가장 중요한 문장: 모델은 함수를 직접 실행하지 않습니다.**
> 모델은 무엇을 어떻게 부를지 **결정만** 하고, **실행은 언제나 우리 코드**가 합니다.
> 그래서 위험한 도구는 안 주면 되고, 실행 전에 값을 검증하거나 차단할 수도 있습니다.

전체 흐름은 다섯 단계입니다.

```
① 함수 만들기        →  ② 사용설명서로 모델에 소개  →  ③ 모델이 호출 요청(tool_calls)
                                                            ↓
⑤ 모델이 최종 답변   ←  ④ 우리가 실행 후 결과를 role:"tool" 로 전달
```

먼저 ① 단계, 도구로 쓸 함수를 만듭니다. (연습이니 진짜 날씨 API 대신 미리 적어둔 값을 돌려줍니다)

> 💡 도구 함수는 어떤 값이 와도 **에러 대신 문자열**을 돌려주게 만듭니다.
> 도중에 에러가 나면 뒤에서 만들 에이전트가 통째로 멈추기 때문입니다.

<!-- manim-visual -->
### 🎬 시각 자료: 함수 호출 5단계

질문 → 모델의 호출 요청(`tool_calls`) → 우리 코드가 실제 실행 → `role:"tool"` 로 결과 전달 → 최종 답변. 모델은 어떤 함수를 어떤 인자로 부를지 정할 뿐, 실행은 언제나 우리 코드가 합니다.

![FunctionCalling](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/main/week2/assets/FunctionCalling.gif)

In [ ]:
def get_weather(city: str) -> str:
    # 실제라면 기상청 API를 부르겠지만, 실습이라 딕셔너리를 '가짜 DB'로 씁니다.
    fake_db = {
        "서울": "맑음, 27도",
        "부산": "흐림, 24도",
        "제주": "비, 22도",
    }
    return fake_db.get(city, f"{city}의 날씨 정보가 없습니다.")

# 모델에 연결하기 전에 함수 단독으로 테스트
print(get_weather("서울"))
print(get_weather("도쿄"))   # 없는 도시

# 💡 fake_db 에 데이터를 더 넣어 다른 데이터로도 실습해 보세요.
#        "대구": "맑음, 29도", "인천": "안개, 23도", "뉴욕": "맑음, 18도",


### ② 도구 스키마 — 모델에게 주는 사용설명서

모델은 우리가 짠 파이썬 코드를 볼 수 없습니다.
그래서 함수의 **이름·기능·필요한 값**을 정해진 형식(JSON)의 **사용설명서**로 만들어 건넵니다.

| 항목 | 의미 |
|---|---|
| `name` | 도구 이름 — **실제 파이썬 함수명과 똑같이** |
| `description` | 무엇을 하는 도구인지 — 모델은 **이 설명만 보고** 쓸지 말지 결정합니다 ⭐ |
| `parameters` → `properties` | 필요한 값들의 이름·타입·설명 (예시를 넣으면 정확도 ↑) |
| `required` | 꼭 필요한 값 목록 (빼먹으면 모델이 인자 없이 부를 수 있음) |

> 💡 도구가 호출되지 않는다면 십중팔구 `description` 이 부실한 탓입니다.
> "특정 도시의 현재 날씨를 알려준다"처럼 **기능과 쓰는 시점**을 구체적으로 쓰세요.

Chat Completions에서는 함수 정보를 `"function"` 키로 한 번 감싸는 형태를 씁니다.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",                          # 실제 파이썬 함수명과 똑같이
            "description": "특정 도시의 현재 날씨를 알려준다.",   # ⭐ 모델은 이 설명만 보고 도구 사용을 결정
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {                               # 함수의 매개변수 이름과 동일하게
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름. 예: 서울, 부산",   # 예시를 넣으면 정확도가 오릅니다
                    },
                },
                "required": ["city"],
            },
        },
    }
]

# 💡 값을 정해진 것 중에서만 고르게 하려면 enum 을 씁니다.
#    "unit": {"type": "string", "enum": ["섭씨", "화씨"], "description": "온도 단위"}

### ③ 도구를 주고 질문하기

`tools=` 인자로 사용설명서를 함께 보내면, 모델이 도구를 쓸지 말지 **스스로 판단**합니다.

- 도구가 필요 없는 질문("안녕?") → 평소처럼 `content` 에 답이 옵니다
- 도구가 필요한 질문("서울 날씨 어때?") → `content` 는 비어 있고(`None`),
  대신 `message.tool_calls` 에 **"이 함수를 이 값으로 실행해 줘"** 요청이 옵니다

`tool_calls` 의 각 항목에는 세 가지가 들어 있습니다.

- `call.function.name` — 부를 함수 이름
- `call.function.arguments` — 넣을 값. ⚠️ 딕셔너리가 아니라 **JSON 문자열**입니다 (`'{"city": "서울"}'`)
- `call.id` — 이 요청의 **번호표**. 결과를 돌려줄 때 어느 요청의 결과인지 짝을 맞추는 용도

In [ ]:
messages = [{"role": "user", "content": "서울 날씨 어때?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,          # ⭐ 도구 설명서를 함께 전달 — function calling 의 시작
)

message = response.choices[0].message

print("일반 답변 content:", message.content)   # 도구를 부르려 할 땐 보통 None
print("함수 호출 요청 tool_calls:")
for call in (message.tool_calls or []):        # 도구를 안 부르면 None 이라 or [] 로 감쌉니다
    print("  호출할 함수:", call.function.name)
    print("  인자(문자열):", call.function.arguments)   # dict 가 아니라 JSON 문자열입니다
    print("  call id:", call.id)

# 💡 content 를 바꿔 모델이 언제 도구를 부르고 언제 안 부르는지 관찰해 보세요.
#    "안녕? 오늘 기분 어때?"            → 도구 없이 바로 답 (tool_calls 가 None)
#    "서울이랑 부산 날씨 둘 다 알려줘"   → tool_calls 가 2개 올 수 있습니다

---
## 8. ④⑤ 함수를 실행하고 결과 돌려주기

모델은 "실행해 줘"라고 **요청**만 한 상태입니다. 아직 날씨 값을 모르니 최종 답변도 못 합니다.
이제 우리가 할 일은 세 가지입니다.

1. 요청받은 함수를 **실제로 실행** (`arguments` 는 JSON 문자열이므로 `json.loads` 로 딕셔너리로 변환)
2. 결과를 **`role:"tool"` 메시지**로 대화에 추가 — `tool_call_id` 에 아까 받은 번호표(`call.id`)를 넣어 짝을 맞춥니다
3. 이 대화를 모델에 **다시** 보내 자연어 최종 답변 받기

즉 **질문 하나에 API 호출이 최소 두 번** — 이 왕복 구조가 function calling의 핵심입니다.

이 시점의 `messages` 는 이렇게 쌓여 있습니다.

```
[0] user      : "서울 날씨 어때?"
[1] assistant : (tool_calls=get_weather(city='서울'))   ← 모델의 호출 요청도 기록에 남깁니다
[2] tool      : "맑음, 27도"  (tool_call_id=call_abc123)
```

> ⚠️ 모델의 호출 요청 메시지(`message`)를 **먼저** append 해야 합니다.
> 요청 없이 결과만 있으면 API가 오류를 냅니다.

In [ ]:
import json

# 1) 모델의 호출 요청 메시지를 먼저 기록에 추가
messages.append(message)

# 2) 요청된 함수를 실행하고 결과를 tool 메시지로 추가
for call in message.tool_calls:
    args = json.loads(call.function.arguments)      # JSON 문자열 -> 딕셔너리
    result = get_weather(**args)                    # get_weather(**{"city":"서울"}) == get_weather(city="서울")
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,                    # 어떤 호출의 결과인지 연결 (필수)
        "content": result,                          # 결과는 반드시 문자열
    })

# 3) 도구 결과가 담긴 대화를 다시 전송 -> 최종 답변
final = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)
print(final.choices[0].message.content)

---
## 9. 에이전트(Agent) — 왕복을 루프로 감싸면 끝

지금까지의 과정을 한 줄로 정리하면:

```
질문 → 모델 → (도구 요청?) → 실행 → 결과 전달 → 모델 → ...
```

8절에서는 이 왕복을 손으로 한 번 했습니다. 하지만 실제 질문은 한 번의 왕복으로 안 끝납니다.
"서울과 부산 날씨 알려주고 25×4도 계산해줘"라면 도구를 **여러 번, 여러 종류** 불러야 하고,
앞 도구의 결과를 보고 다음 도구를 정하기도 합니다.

이 왕복을 **모델이 더 이상 도구를 부르지 않을 때까지 반복**하면 — 그게 곧 **에이전트**입니다.
거창해 보여도 실체는 **20줄짜리 반복문**입니다.

### 준비물 세 가지

| 준비물 | 역할 |
|---|---|
| `tools` | 모델이 읽는 **사용설명서 목록** |
| `available_functions` | 이름(문자열) → 실제 파이썬 함수 **연결표** |
| `max_turns` | **무한루프 방지** 안전장치 |

에이전트다운 모습을 보려면 도구가 둘은 필요하니, 두 번째 도구로 **계산기**를 추가합니다.
LLM은 계산에 약해서 그럴듯한 오답을 내기 쉽기 때문에, 계산은 정확한 도구에 맡기는 것이 정석입니다.

> ⚠️ 계산기 `calculate` 안에서 쓰는 `eval` 은 **파이썬 코드로 그대로 실행하는 함수**라,
> 위험한 코드가 섞여 들어와도 실행해 버립니다. 그래서 숫자와 연산 기호만 통과시켜 걸러 줍니다.
> 이 값을 만드는 건 모델이니, **모델이 준 값도 사용자 입력처럼 검증**해야 합니다.

<!-- manim-visual -->
### 🎬 시각 자료: 에이전트 루프

도구 호출 요청이 없어질 때까지 같은 왕복을 반복합니다. 모델이 스스로 도구를 골라 쓰다가 더 부를 도구가 없으면 최종 답변을 내고 루프가 끝납니다.

![AgentLoop](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/main/week2/assets/AgentLoop.gif)

In [ ]:
def calculate(expression: str) -> str:
    # 실습용 최소 구현. 허용 문자만 통과시킨 뒤 계산합니다.
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "허용되지 않은 문자가 있습니다."
    try:
        return str(eval(expression))
    except Exception as e:
        # 도구는 예외를 밖으로 던지지 않습니다. 에러도 문자열로 주면 모델이 스스로 고쳐 재시도합니다.
        return f"계산 오류: {e}"


# 이름(문자열) -> 실제 함수 연결표. 모델은 "calculate" 라는 이름만 주기 때문에 필요합니다.
available_functions = {
    "get_weather": get_weather,     # () 를 붙이지 않습니다 — 실행이 아니라 함수 자체를 보관
    "calculate": calculate,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨를 알려준다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "도시 이름. 예: 서울"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "사칙연산 수식을 계산한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "계산할 수식. 예: '12 * 8'"},
                },
                "required": ["expression"],
            },
        },
    },
]

# 💡 도구를 더 만들어 보세요. 함수 정의 → available_functions 등록 → tools 스키마 추가, 3단계면 끝입니다.
#    def get_population(city: str) -> str:
#        fake_pop = {"서울": "938만 명", "부산": "329만 명", "제주": "67만 명"}
#        return fake_pop.get(city, f"{city}의 인구 정보가 없습니다.")
#    도구가 늘면 "서울 인구는 부산의 몇 배야?" 처럼 여러 도구를 연달아 써야 하는 질문도 풀립니다.

In [ ]:
import json

def run_agent(user_message, max_turns=5):
    # max_turns: 모델이 도구를 계속 부르며 맴돌 때를 대비한 무한루프 방지 장치.
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message
        messages.append(message)

        # 도구 호출 요청이 없다 = 할 일을 다 했다 -> 최종 답변
        if not message.tool_calls:
            return message.content

        for call in message.tool_calls:
            func = available_functions[call.function.name]   # 이름 -> 실제 함수
            args = json.loads(call.function.arguments)
            result = func(**args)
            print(f"  🔧 {call.function.name}({args}) -> {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })
        # 다음 바퀴로 돌아가 결과를 모델에게 보여주고 다음 행동을 정하게 합니다.

    return "(최대 반복 횟수를 초과했습니다.)"

# 💡 모델이 없는 함수 이름을 지어내면 KeyError 가 납니다. 아래처럼 막을 수 있습니다.
#    func = available_functions.get(call.function.name)
#    result = func(**args) if func else f"{call.function.name} 이라는 도구는 없습니다."


에이전트를 불러 봅니다. 아래 질문은 **날씨 조회 2번 + 계산 1번**을 모두 해야 풀립니다.
어떤 도구를 어떤 순서로 쓰는지 `🔧` 로그로 지켜보세요.
**아무 지시도 하지 않았는데 모델이 스스로 판단**한다는 것이 핵심입니다.

> ⚠️ 무료 모델이라 도구를 한 번에 완벽히 못 쓸 때도 있습니다.
> 그럴 땐 ① 질문을 더 명확히 ② `description` 을 더 구체적으로 ③ 그냥 다시 실행 — 해보세요.
> 이런 실패를 관찰하고 고치는 과정 자체가 실습입니다.

In [ ]:
# 도구 2개(get_weather, calculate)를 모두 써야 풀리는 질문입니다.
answer = run_agent("서울과 부산 날씨를 알려주고, 25 곱하기 4가 얼마인지도 계산해줘.")
print("\n최종 답변:\n", answer)

# 💡 다른 질문으로도 실습해 보세요. 🔧 로그로 도구 호출 횟수를 비교할 수 있습니다.
#    run_agent("안녕! 너는 누구야?")                → 도구를 아예 안 부름
#    run_agent("서울 기온에 2를 곱하면 몇 도야?")   → 날씨 결과를 계산 도구로 연결(도구 연쇄)

---
## 10. 연습문제 🎯

지금까지의 `get_weather` 는 미리 적어둔 **가짜 날씨**였습니다. 밖에 비가 와도 서울은 언제나 "맑음, 27도"였죠.

이제 이 도구를 **진짜 날씨 API**로 업그레이드해서, 에이전트가 지금 이 순간의 실제 데이터로 답하게 만듭니다.

날씨 API는 [**Open-Meteo**](https://open-meteo.com) 를 씁니다. **가입도 API 키도 필요 없는** 무료 서비스입니다.

<br>
문제는 세 개이고, 하나로 이어지는 이야기입니다.

| 문제 | 하는 일 | 여러분이 할 일 | 빈칸이 담고 있는 핵심 |
|---|---|---|---|
| **1** | 진짜 날씨 도구를 에이전트에 연결하기 | 빈칸 1개 ✏️ | **도구의 `description` 이 성능을 좌우한다** |
| **2** | 에이전트 루프 완성하기 | 빈칸 1개 ✏️ | **모델은 함수를 실행하지 않는다 — 실행은 우리 코드가** |
| **3** | 에이전트에게 페르소나(성격) 입히기 | 프롬프트 실험 👀 | system 메시지는 행동 규칙을 정한다 |

<br>
<br>
진행 방법은 간단합니다.

- `# TODO` 가 붙은 곳만 채우면 됩니다. 나머지 코드는 그대로 실행하세요.
- 막히면 빈칸 바로 아래의 "정답 보기"를 펼쳐서 확인하세요. 정답을 보고 따라 써도 충분히 공부가 됩니다!

### 문제 1. 진짜 날씨 도구 `get_real_weather` 를 에이전트에 추가하기

새 도구를 붙이는 순서는 7절에서와 언제나 같습니다.

**① 함수 만들기 → ② 이름표에 등록 → ③ 사용설명서(스키마) 추가 → ④ 테스트**

<br>

**1단계. 함수 만들기**

아래 함수는 Open-Meteo에 물어봐서 실제 날씨를 가져옵니다. 코드는 완성되어 있으니, 실행하기 전에 마지막 `return` 한 줄만 눈여겨보세요.

> ⭐ 모델은 이 함수가 돌려준 **문자열만 보고** 답을 만듭니다.
> `"23"` 만 돌려주면 모델은 그게 기온인지 습도인지 알 수 없습니다.
> 그래서 `"서울: 기온 23°C, 습도 60%"` 처럼 **이름표와 단위가 붙은 문장**으로 돌려줍니다.

In [ ]:
import requests

def get_real_weather(city: str) -> str:
    # Open-Meteo API로 도시의 '진짜' 현재 날씨를 조회한다. (무료, API 키 불필요)
    try:
        # 1) 도시 이름 -> 위도/경도 (한국어 이름도 가능)
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "ko"},
            timeout=5,
        ).json()
        if not geo.get("results"):
            return f"'{city}' 도시를 찾을 수 없습니다."
        loc = geo["results"][0]

        # 2) 위도/경도 -> 현재 날씨
        w = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": loc["latitude"],
                "longitude": loc["longitude"],
                "current": "temperature_2m,apparent_temperature,relative_humidity_2m,wind_speed_10m",
            },
            timeout=5,
        ).json()["current"]
    except requests.exceptions.RequestException as e:
        return f"네트워크 오류: {e}"

    # ⭐ 모델은 아래 문자열'만' 보고 답을 만듭니다.
    #    그래서 값마다 이름표(기온·습도·풍속)와 단위(°C, %, km/h)를 붙여서 돌려줍니다.
    return f"{loc['name']}: 기온 {w['temperature_2m']}°C (체감 {w['apparent_temperature']}°C), 습도 {w['relative_humidity_2m']}%, 풍속 {w['wind_speed_10m']}km/h"

# 진짜 오늘 날씨가 나오는지 확인해 보세요.
print(get_real_weather("서울특별시"))
print(get_real_weather("Busan"))
print(get_real_weather("없는도시123"))   # 오류 처리 확인

# 💡 다른 도시로도 실습해 보세요. 한국어·영어 이름 모두 됩니다.
#    print(get_real_weather("제주"));  print(get_real_weather("New York"))

**2단계. 함수를 이름표에 등록**

모델은 함수를 직접 실행하지 못합니다. "`get_real_weather` 실행해줘"라는 **이름**(**문자열**)을 보낼 뿐이죠.

그 이름으로 진짜 파이썬 함수를 찾아주는 연결표가 9절의 `available_functions` 딕셔너리였습니다.

새 도구를 연결표에 등록하세요. 가짜 `get_weather` 는 헷갈리지 않게 치워 둡시다.

In [ ]:
# 새 도구 등록: 왼쪽(따옴표 있음)은 모델이 보내는 '이름', 오른쪽(따옴표 없음)은 함수 그 자체.
# () 를 붙이면 지금 당장 실행되어 버리니 붙이지 않습니다.
available_functions["get_real_weather"] = get_real_weather

# 가짜 날씨 도구 제거
available_functions.pop("get_weather", None)
tools[:] = [t for t in tools if t["function"]["name"] != "get_weather"]

# dict_keys(['calculate', 'get_real_weather']) 가 보이면 성공!
print(available_functions.keys())

**3단계. 도구 스키마 추가 — ✏️ 빈칸 1개**

모델은 우리가 짠 파이썬 코드를 보지 못합니다. **스키마에 적은 설명만 읽고** 도구를 쓸지, 인자를 뭐라고 채울지 결정합니다.

<br>

여러분이 채울 곳은 `description` 하나입니다. 이 한 문장이 **모델이 도구를 쓸지 판단하는 유일한 근거**입니다.

- 잘 쓰는 요령: 이 도구가 **무엇을** 하는지 구체적으로. (예: "특정 도시의 ~를 조회한다.")
- 도구가 호출되지 않는다면 십중팔구 이 설명이 부실한 탓입니다 — 오늘 수업에서 **가장 기억해야 할 문장** 중 하나입니다.


`name` 은 미리 채워 두었습니다. 2단계에서 등록한 이름과 **한 글자도 다르지 않아야** 한다는 점만 눈여겨보세요.

In [ ]:
tools.append({
    "type": "function",
    "function": {
        "name": "get_real_weather",   # 2단계에서 등록한 이름과 한 글자도 다르지 않게 (미리 채워 두었습니다)
        "description": "________",  # TODO ✏️: 이 도구가 무엇을 하는지 한 문장으로 쓰세요. 모델은 이 설명'만' 보고 도구 사용을 결정합니다!
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "날씨를 조회할 도시 이름. 예: 서울, Busan",   # 예시 값을 넣으면 모델이 인자를 더 정확히 채웁니다
                },
            },
            "required": ["city"],
        },
    },
})

# ['calculate', 'get_real_weather'] 가 보이면 성공!
print([t["function"]["name"] for t in tools])

<details>
<summary> 3단계 정답 보기</summary>

```python
"description": "특정 도시의 실제 현재 날씨(기온·습도·풍속)를 조회한다.",
```

문구가 달라도 됩니다. 다만 "날씨를 조회한다"는 **기능**이 분명히 드러나야 모델이 날씨 질문에서 이 도구를 골라 씁니다.
빈 문자열이나 "도구입니다" 같은 설명으로 두면 4단계에서 모델이 이 도구를 **아예 부르지 않을 수도** 있습니다 — 그게 바로 오늘의 교훈입니다.

</details>

**4단계. 테스트!**

9절의 `run_agent` 는 `tools` 와 `available_functions` 를 그대로 읽기 때문에, 방금 등록한 새 도구를 바로 씁니다.

✅ **성공 기준**: 🔧 로그에 `get_real_weather` 가 찍히고, 답변 속 기온이 진짜 오늘 날씨와 비슷하면 성공입니다. 창밖과 비교해 보세요! ☀️

In [ ]:
print(run_agent("서울의 지금 실제 날씨를 알려주고, 현재 기온에 2를 곱하면 얼마인지도 계산해줘."))

# 💡 다른 질문으로도 실습해 보세요.
#    run_agent("서울과 제주 중 어디가 더 습해?")        → 날씨 조회 2번 후 비교
#    run_agent("오늘 서울에서 우산이 필요할까?")        → 조회 값을 근거로 판단

### 문제 2. 에이전트 루프 완성하기 — ✏️ 빈칸 1개

9절에서 만든 에이전트 루프를 다시 만납니다. 이번 버전(`run_agent_v2`)은 문제 3을 위해 **system 프롬프트를 받는 기능**이 하나 추가되었습니다.

```
질문 → 모델 → 도구를 부르고 싶어 하나?
                │
                ├─ 아니오 → ① 최종 답변, 끝!
                │
                └─   예   → ② 이름으로 진짜 함수를 찾아 실행 ← ✏️ 여기가 빈칸
                                  ↓
                               ③ 결과를 모델에게 돌려주기 → 다시 모델에게 (반복)
```

<br>

②가 빈칸인 이유가 있습니다. 오늘 수업에서 가장 중요한 문장이 바로 이 줄에 들어 있기 때문입니다.

> ⭐ **모델은 함수를 직접 실행하지 않습니다.** 모델은 `"get_real_weather"` 라는 **이름**(**문자열**)을 보낼 뿐이고,
> 그 이름으로 진짜 함수를 찾아 **실행하는 것은 언제나 우리 코드**입니다.

> ⚠️ 빈칸을 채우기 전에 실행하면 오류가 납니다. 정상이니 당황하지 마세요!

✅ **성공 기준**: 실행했을 때 🔧 로그가 **두 줄** 찍히면 성공 — 모델이 스스로 도구를 두 번 부른 것입니다.

In [ ]:
import json

def run_agent_v2(user_message, system_prompt="", max_turns=5):
    messages = []
    if system_prompt:   # ✨ 추가된 기능: system 프롬프트가 있으면 맨 앞에 넣습니다 (문제 3에서 사용)
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    for turn in range(max_turns):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        messages.append(message)

        # 핵심 ①: 도구 호출 요청(tool_calls)이 없다 = 할 일을 다 했다 → 최종 답변, 끝!
        if not message.tool_calls:
            return message.content

        for call in message.tool_calls:
            # 핵심 ②: 모델은 이름(문자열)만 보냅니다. 연결표에서 진짜 함수를 찾아 실행하는 건 우리 몫!
            # TODO ✏️: 모델이 보낸 함수 이름으로 연결표(available_functions)에서 진짜 함수를 꺼내세요.
            #   힌트: 모델이 보낸 이름은 call.function.name 에 들어 있습니다. (9절 참고)
            func = available_functions[________]
            args = json.loads(call.function.arguments)
            result = func(**args)
            print(f"  🔧 {call.function.name}({args}) -> {result}")

            # 핵심 ③: 실행 결과를 role:"tool" 메시지로 돌려줍니다. (8절)
            #         tool_call_id 는 어느 호출의 결과인지 짝을 맞추는 '번호표'(call.id)입니다.
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })
        # 다음 바퀴: 결과를 본 모델이 다음 행동을 정합니다.

    return "(최대 반복 횟수를 초과했습니다.)"

# 이 질문은 도구를 두 번 불러야 답할 수 있습니다. 🔧 로그가 두 줄 찍히는지 관찰하세요.
print(run_agent_v2("서울이랑 부산 중에 지금 어디가 더 따뜻해?"))

# 💡 도전 과제(선택): 핵심 ①③ 자리도 지우고 직접 다시 채워 보세요. 채울 수 있다면 에이전트를 완전히 이해한 것입니다!

<details>
<summary> 문제 2 정답 보기</summary>

```python
func = available_functions[call.function.name]
```

모델이 보낸 것은 `"get_real_weather"` 라는 **글자**일 뿐입니다. 그 글자를 열쇠 삼아 연결표에서 진짜 파이썬 함수를 꺼내고, 실행은 바로 다음 줄에서 **우리 코드**(`func(**args)`)가 합니다.

이 한 줄이 "모델은 함수를 실행하지 않는다 — 결정만 하고, 실행은 언제나 우리 코드가 한다"는 오늘 수업의 핵심 문장 그 자체입니다.

</details>

### 문제 3. 기상 캐스터 페르소나 만들기 (프롬프트 실험 👀)

이번에는 채울 코드가 없습니다. 완성된 프롬프트를 실행해 보고, **직접 바꿔 보는** 문제입니다.

5절에서 배운 `system` 메시지는 말투뿐 아니라 에이전트의 **행동 규칙**을 정하는 곳이기도 합니다.
아래 프롬프트에는 세 가지 규칙이 들어 있습니다.

    ① 모든 답변을 '📺 날씨 캐스터입니다.'로 시작하기 — 형식 규칙
    ② 날씨에 맞는 옷차림 조언을 한 문장 덧붙이기 — 역할 규칙
    ③ 도구로 조회한 실제 데이터만 근거로 말하기 — 행동 규칙

특히 ③은 에이전트가 정보를 지어내는 것(환각)을 막기 위해 실제 서비스에서도 널리 쓰이는 방법입니다.

In [ ]:
# 세 규칙(①형식 ②역할 ③행동)이 담긴 system 프롬프트입니다. 그대로 실행해 보세요.
weather_caster_prompt = """너는 TV 날씨 캐스터다.
- 모든 답변을 '📺 날씨 캐스터입니다.' 로 시작한다.
- 날씨에 맞는 옷차림 조언을 한 문장 덧붙인다.
- 반드시 도구로 조회한 실제 데이터만 근거로 말한다. 조회하지 않은 값은 지어내지 말고 모른다고 답한다."""

# ✅ 성공 기준: 답이 '📺 날씨 캐스터입니다.' 로 시작하고, 실제 날씨 + 옷차림 조언이 나오면 성공!
print(run_agent_v2("서울 날씨 어때? 오늘 뭐 입고 나갈까?", system_prompt=weather_caster_prompt))

# ✏️ 이제 프롬프트를 직접 바꿔 실험해 보세요. 프롬프트에는 정답이 없습니다!
#    "너는 등산 가이드다. 날씨를 확인한 뒤 오늘 산행이 적절한지 조언해라."
# 💡 규칙 ③이 지켜지는지 보려면 조회하지 않은 값을 물어보세요.
#    run_agent_v2("서울의 내일 오후 강수 확률은?", system_prompt=weather_caster_prompt)

---
## 정리

### 이번 주에 배운 것

| 주제 | 핵심 |
|---|---|
| **OpenRouter** | `base_url` 만 바꾸면 OpenAI 라이브러리로 무료 모델을 쓸 수 있다 |
| **Chat Completions** | `messages` 로 요청하고 `choices[0].message.content` 로 응답을 받는다 |
| **Responses API** | `input` / `output_text` / `previous_response_id` — 더 단순하지만 개념은 동일 |
| **system 메시지** | 모델의 역할·말투·출력 형식을 정하는 보이지 않는 지시문 |
| **temperature** | 낮으면 일관적(사실·코드), 높으면 창의적(창작·아이디어) |
| **대화 맥락** | API는 상태를 유지하지 않는다. `assistant` 답변까지 직접 누적해야 맥락이 이어진다 |
| **Function calling** | 스키마로 도구 정의 → 모델의 `tool_calls` 요청 → **코드가 실행** → `role:"tool"` 로 결과 전달 |
| **에이전트 루프** | 도구 호출이 없어질 때까지 반복하는 구조 |
| **외부 API 연동** | Open-Meteo 실시간 날씨를 도구로 연결. 반환값의 형태, 스키마 설명, 행동 규칙이 에이전트 품질을 결정한다 |

### 반드시 기억할 세 가지

1. **모델은 함수를 실행하지 않는다.** 호출 대상을 결정할 뿐, 실행은 항상 코드가 담당한다.
2. **모델의 기억은 요청마다 구성된다.** `messages` 에 없는 정보는 모델에게 존재하지 않는다.
3. **도구의 `description` 이 에이전트의 성능을 좌우한다.** 설명이 부실하면 도구는 호출되지 않는다.

### 추가 학습 과제

- 연습문제의 Open-Meteo와 같이 **다른 공개 API**(환율·뉴스·대중교통 등)를 도구로 연결해 보기
- 도구를 3~4개로 늘리고, **여러 도구를 연달아 사용해야 풀리는 질문** 실험해 보기
- 동일한 코드를 **다른 모델**로 실행해 도구 사용 능력 비교하기
- 웹 검색·코드 실행 같은 **내장 도구(built-in tools)** 와 여러 에이전트를 엮는 **Agents SDK** 살펴보기

수고하셨습니다! 🎉